In [1]:
from google.colab import files
uploaded = files.upload()

Saving kaggle.json to kaggle.json


In [2]:
# Create Kaggle folder and set up authorization key
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download the NYC Taxi Trip Duration dataset
!kaggle competitions download -c nyc-taxi-trip-duration

# Unzip all downloaded files
!unzip -o nyc-taxi-trip-duration.zip

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/DownloadDataFiles
unzip:  cannot find or open nyc-taxi-trip-duration.zip, nyc-taxi-trip-duration.zip.zip or nyc-taxi-trip-duration.zip.ZIP.


In [3]:
# Create Kaggle directory and move credentials
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download direct dataset (no competition rules required)
!kaggle datasets download -d yasserh/nyc-taxi-trip-duration

# Unzip the downloaded file
!unzip -o nyc-taxi-trip-duration.zip

Dataset URL: https://www.kaggle.com/datasets/yasserh/nyc-taxi-trip-duration
License(s): CC0-1.0
100% 63.9M/63.9M [00:00<00:00, 124MB/s]

Archive:  nyc-taxi-trip-duration.zip
  inflating: NYC.csv                 


In [4]:
import pandas as pd

# Load the unzipped CSV dataset
df = pd.read_csv('NYC.csv')

# Display row count, column count, and first 5 rows
print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (1458644, 11)


,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


In [5]:
# Convert pickup_datetime to actual pandas datetime objects
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Extract temporal features needed for spatio-temporal forecasting
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['day_name'] = df['pickup_datetime'].dt.day_name()
df['month'] = df['pickup_datetime'].dt.month
df['date_hour'] = df['pickup_datetime'].dt.floor('h')  # Group by exact hour bucket

# Filter out geographical outliers (keep records strictly within NYC bounds)
nyc_lat_range = (40.57, 40.91)
nyc_lon_range = (-74.15, -73.70)

df_clean = df[
    (df['pickup_latitude'].between(*nyc_lat_range)) &
    (df['pickup_longitude'].between(*nyc_lon_range))
]

print("Original records:", len(df))
print("Cleaned records within NYC bounds:", len(df_clean))
df_clean[['pickup_datetime', 'hour', 'day_name', 'pickup_latitude', 'pickup_longitude']].head()

Original records: 1458644
Cleaned records within NYC bounds: 1458155


,pickup_datetime,hour,day_name,pickup_latitude,pickup_longitude
0,2016-03-14 17:24:55,17,Monday,40.767937,-73.982155
1,2016-06-12 00:43:35,0,Sunday,40.738564,-73.980415
2,2016-01-19 11:35:24,11,Tuesday,40.763939,-73.979027
3,2016-04-06 19:32:31,19,Wednesday,40.719971,-74.010040
4,2016-03-26 13:30:55,13,Saturday,40.793209,-73.973053


In [6]:
# Convert pickup_datetime to actual pandas datetime objects
df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

# Extract temporal features needed for spatio-temporal forecasting
df['hour'] = df['pickup_datetime'].dt.hour
df['day_of_week'] = df['pickup_datetime'].dt.dayofweek
df['day_name'] = df['pickup_datetime'].dt.day_name()
df['month'] = df['pickup_datetime'].dt.month
df['date_hour'] = df['pickup_datetime'].dt.floor('h')  # Group by exact hour bucket

# Filter out geographical outliers (keep records strictly within NYC bounds)
nyc_lat_range = (40.57, 40.91)
nyc_lon_range = (-74.15, -73.70)

df_clean = df[
    (df['pickup_latitude'].between(*nyc_lat_range)) &
    (df['pickup_longitude'].between(*nyc_lon_range))
]

print("Original records:", len(df))
print("Cleaned records within NYC bounds:", len(df_clean))
df_clean[['pickup_datetime', 'hour', 'day_name', 'pickup_latitude', 'pickup_longitude']].head()

Original records: 1458644
Cleaned records within NYC bounds: 1458155


,pickup_datetime,hour,day_name,pickup_latitude,pickup_longitude
0,2016-03-14 17:24:55,17,Monday,40.767937,-73.982155
1,2016-06-12 00:43:35,0,Sunday,40.738564,-73.980415
2,2016-01-19 11:35:24,11,Tuesday,40.763939,-73.979027
3,2016-04-06 19:32:31,19,Wednesday,40.719971,-74.010040
4,2016-03-26 13:30:55,13,Saturday,40.793209,-73.973053


In [7]:
import numpy as np

# 1. Divide NYC map into a 20x20 grid of spatial zones
lat_bins = np.linspace(40.57, 40.91, 20)
lon_bins = np.linspace(-74.15, -73.70, 20)

# 2. Assign each pickup location to its corresponding grid coordinates
df_clean['lat_grid'] = pd.cut(df_clean['pickup_latitude'], bins=lat_bins, labels=False)
df_clean['lon_grid'] = pd.cut(df_clean['pickup_longitude'], bins=lon_bins, labels=False)

# 3. Combine grid coordinates into a unique Zone ID (e.g., '12_8')
df_clean['zone_id'] = df_clean['lat_grid'].astype(str) + "_" + df_clean['lon_grid'].astype(str)

# 4. Count total ride requests (demand) per zone per hour
df_demand = df_clean.groupby(['date_hour', 'zone_id', 'hour', 'day_of_week']).size().reset_index(name='demand')

print("Aggregated Demand Dataset Shape:", df_demand.shape)
df_demand.head(10)

/tmp/ipykernel_1053/85380538.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['lat_grid'] = pd.cut(df_clean['pickup_latitude'], bins=lat_bins, labels=False)
/tmp/ipykernel_1053/85380538.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['lon_grid'] = pd.cut(df_clean['pickup_longitude'], bins=lon_bins, labels=False)
/tmp/ipykernel_1053/85380538.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = val

Aggregated Demand Dataset Shape: (116432, 5)


,date_hour,zone_id,hour,day_of_week,demand
0,2016-01-01,10_6,0,4,49
1,2016-01-01,10_7,0,4,70
2,2016-01-01,10_8,0,4,8
3,2016-01-01,10_9,0,4,4
4,2016-01-01,11_10,0,4,1
5,2016-01-01,11_11,0,4,2
6,2016-01-01,11_13,0,4,1
7,2016-01-01,11_6,0,4,15
8,2016-01-01,11_7,0,4,42
9,2016-01-01,11_8,0,4,41


In [8]:
# Sort by zone and timestamp to calculate accurate sequential time lags
df_demand = df_demand.sort_values(['zone_id', 'date_hour']).reset_index(drop=True)

# Create lag features for historical demand context
df_demand['lag_1h'] = df_demand.groupby('zone_id')['demand'].shift(1)   # Demand 1 hour ago
df_demand['lag_2h'] = df_demand.groupby('zone_id')['demand'].shift(2)   # Demand 2 hours ago
df_demand['lag_24h'] = df_demand.groupby('zone_id')['demand'].shift(24) # Demand 24 hours ago (same time yesterday)

# Drop rows with NaN values created by shifting
df_model = df_demand.dropna().reset_index(drop=True)

print("Feature Engineering Complete! Training Dataset Shape:", df_model.shape)
df_model[['date_hour', 'zone_id', 'hour', 'day_of_week', 'lag_1h', 'lag_24h', 'demand']].head()

Feature Engineering Complete! Training Dataset Shape: (113151, 8)


,date_hour,zone_id,hour,day_of_week,lag_1h,lag_24h,demand
0,2016-01-08 21:00:00,10_10,21,4,1.0,1.0,1
1,2016-01-08 22:00:00,10_10,22,4,1.0,1.0,1
2,2016-01-09 02:00:00,10_10,2,5,1.0,1.0,1
3,2016-01-09 09:00:00,10_10,9,5,1.0,1.0,1
4,2016-01-09 10:00:00,10_10,10,5,1.0,1.0,1


In [9]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Define feature columns (inputs) and target variable (demand)
features = ['hour', 'day_of_week', 'lag_1h', 'lag_2h', 'lag_24h']
X = df_model[features]
y = df_model['demand']

# Split dataset into training (80%) and test (20%) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train the Demand Prediction Engine using Random Forest
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Evaluate predictions
y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print(f"Model Trained Successfully!")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")

# Display sample actual vs predicted values
results = pd.DataFrame({'Actual Demand': y_test, 'Predicted Demand': np.round(y_pred)}).head(10)
results

Model Trained Successfully!
Mean Absolute Error (MAE): 2.94
Root Mean Squared Error (RMSE): 5.00


,Actual Demand,Predicted Demand
76640,8,5.0
47506,5,2.0
109621,18,24.0
31383,32,34.0
18086,3,5.0
111018,1,1.0
91325,5,2.0
41531,2,9.0
15923,1,2.0
13048,2,1.0


In [10]:
import joblib

# 1. Generate demand predictions across all zones in the dataset
df_model['predicted_demand'] = np.round(model.predict(df_model[features]))

# 2. Define Driver Repositioning Logic
def generate_recommendation(predicted_demand):
    if predicted_demand >= 20:
        return 'CRITICAL HOTSPOT: High driver allocation needed'
    elif predicted_demand >= 10:
        return 'MODERATE DEMAND: Reposition available drivers'
    else:
        return 'LOW DEMAND: Normal coverage'

df_model['recommendation'] = df_model['predicted_demand'].apply(generate_recommendation)

# 3. Save the trained model file for the Web Dashboard backend
joblib.dump(model, 'demand_model.pkl')

print("Driver Recommendation Service Logic Implemented & Model Saved!")
df_model[['date_hour', 'zone_id', 'predicted_demand', 'recommendation']].tail(10)

Driver Recommendation Service Logic Implemented & Model Saved!


,date_hour,zone_id,predicted_demand,recommendation
113141,2016-06-30 05:00:00,9_9,2.0,LOW DEMAND: Normal coverage
113142,2016-06-30 06:00:00,9_9,3.0,LOW DEMAND: Normal coverage
113143,2016-06-30 08:00:00,9_9,2.0,LOW DEMAND: Normal coverage
113144,2016-06-30 09:00:00,9_9,2.0,LOW DEMAND: Normal coverage
113145,2016-06-30 10:00:00,9_9,1.0,LOW DEMAND: Normal coverage
113146,2016-06-30 12:00:00,9_9,2.0,LOW DEMAND: Normal coverage
113147,2016-06-30 13:00:00,9_9,1.0,LOW DEMAND: Normal coverage
113148,2016-06-30 20:00:00,9_9,1.0,LOW DEMAND: Normal coverage
113149,2016-06-30 21:00:00,9_9,1.0,LOW DEMAND: Normal coverage
113150,2016-06-30 23:00:00,9_9,1.0,LOW DEMAND: Normal coverage


In [13]:
import folium
from folium.plugins import HeatMap

# 1. Map zone grid IDs back to center latitude and longitude coordinates
lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
lon_centers = (lon_bins[:-1] + lon_bins[1:]) / 2

df_model['lat_idx'] = df_model['zone_id'].apply(lambda x: int(x.split('_')[0]))
df_model['lon_idx'] = df_model['zone_id'].apply(lambda x: int(x.split('_')[1]))

df_model['latitude'] = df_model['lat_idx'].apply(lambda x: lat_centers[x] if x < len(lat_centers) else lat_centers[-1])
df_model['longitude'] = df_model['lon_idx'].apply(lambda x: lon_centers[x] if x < len(lon_centers) else lon_centers[-1])

# 2. Select a snapshot to visualize hotspots
latest_time = df_model['date_hour'].max()
snapshot_df = df_model[df_model['date_hour'] == latest_time]

# 3. Initialize Folium Map using Esri WorldStreetMap (Reliable free tiles for Colab)
nyc_map = folium.Map(
    location=[40.7128, -74.0060],
    zoom_start=11,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',
    attr='Esri'
)

# 4. Extract heat data (latitude, longitude, predicted demand)
heat_data = [[row['latitude'], row['longitude'], row['predicted_demand']] for index, row in snapshot_df.iterrows()]

# 5. Add HeatMap layer to the dashboard map
HeatMap(heat_data, radius=15, blur=10, max_zoom=1).add_to(nyc_map)

print(f"Heatmap Dashboard generated for time slot: {latest_time}")
nyc_map

Heatmap Dashboard generated for time slot: 2016-06-30 23:00:00
